# **Exercícios Aula 5**: Inferência Estatística

----

Dataset:

Titanic

Objetivo: Aplicar testes estatísticos para responder perguntas a partir de dados reais.

- O .csv pode ser adquirido via: https://www.kaggle.com/datasets/brendan45774/test-file

-----------------

In [0]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Carregar o dataset Titanic
df = pd.read_csv('./data/tested.csv')

print(f'Shape: {df.shape}')
print(f'\nColunas: {df.columns.tolist()}')
print(f'\nValores nulos:\n{df.isnull().sum()}')
display(df.head())

## EXERCÍCIO 1

A variável 'Fare' segue distribuição normal?

- H0: normal
- H1: não-normal

In [0]:
# EXERCÍCIO 1: Teste de Normalidade para 'Fare'
# H0: Fare segue distribuicao normal
# H1: Fare nao segue distribuicao normal

fare_data = df['Fare'].dropna()

# Teste Shapiro-Wilk
stat, p_value = stats.shapiro(fare_data)

print('=== Teste de Normalidade (Shapiro-Wilk) ===')
print(f'Estatistica W: {stat:.4f}')
print(f'p-valor: {p_value:.6e}')
print(f'Nivel de significancia (a): 0.05')

if p_value < 0.05:
    print(f'\nComo p-valor ({p_value:.6e}) < a (0.05), rejeitamos H0.')
    print('Conclusao: Fare NAO segue distribuicao normal.')
else:
    print(f'\nComo p-valor ({p_value:.6e}) >= a (0.05), nao rejeitamos H0.')
    print('Conclusao: Fare segue distribuicao normal.')

# Visualizacao - Histograma e Q-Q Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(fare_data, bins=30, edgecolor='black', color='steelblue')
axes[0].set_title('Distribuicao de Fare')
axes[0].set_xlabel('Fare')
axes[0].set_ylabel('Frequencia')

stats.probplot(fare_data, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot - Fare')

plt.tight_layout()
plt.show()

-----------------

## EXERCÍCIO 2

Teste A/B: Sobrevivência como métrica binária

Crie um A/B onde:
- A: passageiros com Fare >= mediana
- B: passageiros com Fare < mediana

In [0]:
# EXERCÍCIO 2: Teste A/B - Sobrevivência por mediana de Fare
# Grupo A: Fare >= mediana
# Grupo B: Fare < mediana
# Métrica: Survived (binária)

mediana_fare = df['Fare'].median()
print(f'Mediana de Fare: {mediana_fare:.4f}')

# Criar grupos
grupo_a = df[df['Fare'] >= mediana_fare]  # Fare >= mediana
grupo_b = df[df['Fare'] < mediana_fare]   # Fare < mediana

# Taxas de sobrevivência
surv_a = grupo_a['Survived'].mean()
surv_b = grupo_b['Survived'].mean()

print(f'\nGrupo A (Fare >= mediana): n={len(grupo_a)}, sobrevivência={surv_a:.4f} ({surv_a*100:.1f}%)')
print(f'Grupo B (Fare < mediana):  n={len(grupo_b)}, sobrevivência={surv_b:.4f} ({surv_b*100:.1f}%)')

# Tabela de contingência
tabela = pd.crosstab(df['Fare'] >= mediana_fare, df['Survived'])
tabela.index = ['B (Fare < mediana)', 'A (Fare >= mediana)']
tabela.columns = ['Não sobreviveu', 'Sobreviveu']
print(f'\nTabela de Contingência:\n{tabela}')

# Teste qui-quadrado de independência
chi2, p_value, dof, expected = stats.chi2_contingency(tabela)

print(f'\n=== Teste Qui-Quadrado de Independência ===')
print(f'Estatística χ²: {chi2:.4f}')
print(f'p-valor: {p_value:.6f}')
print(f'Graus de liberdade: {dof}')

if p_value < 0.05:
    print(f'\nComo p-valor ({p_value:.6f}) < α (0.05), rejeitamos H0.')
    print('Conclusão: Há diferença significativa na sobrevivência entre os grupos.')
else:
    print(f'\nComo p-valor ({p_value:.6f}) >= α (0.05), não rejeitamos H0.')
    print('Conclusão: Não há diferença significativa na sobrevivência entre os grupos.')

-----------------

## EXERCÍCIO 3

O time de análise quer verificar se passageiros da 1ª classe sobreviveram mais do que passageiros da 3ª classe.

Vamos estruturar isso como um Teste A/B observacional:

- Grupo A (Controle): Passageiros da 3ª classe

- Grupo B (Tratamento): Passageiros da 1ª classe

Métrica: Survived (0 = não sobreviveu, 1 = sobreviveu)

In [0]:
# EXERCÍCIO 3: Teste A/B - 1ª classe vs 3ª classe (Sobrevivência)
# Grupo A (Controle): Passageiros da 3ª classe (Pclass = 3)
# Grupo B (Tratamento): Passageiros da 1ª classe (Pclass = 1)
# Métrica: Survived (binária)

df_ab = df[df['Pclass'].isin([1, 3])]

grupo_a = df_ab[df_ab['Pclass'] == 3]  # 3ª classe (controle)
grupo_b = df_ab[df_ab['Pclass'] == 1]  # 1ª classe (tratamento)

surv_a = grupo_a['Survived'].mean()
surv_b = grupo_b['Survived'].mean()

print(f'Grupo A (3ª classe): n={len(grupo_a)}, sobrevivência={surv_a:.4f} ({surv_a*100:.1f}%)')
print(f'Grupo B (1ª classe): n={len(grupo_b)}, sobrevivência={surv_b:.4f} ({surv_b*100:.1f}%)')

# Tabela de contingência
tabela = pd.crosstab(df_ab['Pclass'], df_ab['Survived'])
tabela.index = ['1ª classe (B)', '3ª classe (A)']
tabela.columns = ['Não sobreviveu', 'Sobreviveu']
print(f'\nTabela de Contingência:\n{tabela}')

# Teste qui-quadrado de independência
chi2, p_value, dof, expected = stats.chi2_contingency(tabela)

print(f'\n=== Teste Qui-Quadrado de Independência ===')
print(f'Estatística χ²: {chi2:.4f}')
print(f'p-valor: {p_value:.6f}')
print(f'Graus de liberdade: {dof}')

if p_value < 0.05:
    print(f'\nComo p-valor ({p_value:.6f}) < α (0.05), rejeitamos H0.')
    print('Conclusão: Há diferença significativa na sobrevivência entre 1ª e 3ª classe.')
    if surv_b > surv_a:
        print('Passageiros da 1ª classe sobreviveram significativamente mais do que os da 3ª classe.')
else:
    print(f'\nComo p-valor ({p_value:.6f}) >= α (0.05), não rejeitamos H0.')
    print('Conclusão: Não há diferença significativa na sobrevivência entre as classes.')

----------------

## EXERCÍCIO 4

O time quer verificar se passageiros mais jovens gastaram mais em tarifas do que passageiros mais velhos.

Vamos estruturar um Teste A/B observacional:

- Grupo A (Controle): Passageiros com idade maior ou igual à mediana

- Grupo B (Tratamento): Passageiros com idade menor que a mediana

Métrica: Fare

In [0]:
# EXERCÍCIO 4: Teste A/B - Jovens vs Idosos (Fare)
# Grupo A (Controle): Idade >= mediana
# Grupo B (Tratamento): Idade < mediana
# Métrica: Fare (contínua)

mediana_age = df['Age'].median()
print(f'Mediana de Age: {mediana_age:.1f} anos')

# Criar grupos
grupo_a = df[df['Age'] >= mediana_age]['Fare'].dropna()  # Mais velhos (controle)
grupo_b = df[df['Age'] < mediana_age]['Fare'].dropna()   # Mais jovens (tratamento)

print(f'\nGrupo A (Age >= mediana): n={len(grupo_a)}, Fare médio={grupo_a.mean():.2f}, mediana={grupo_a.median():.2f}')
print(f'Grupo B (Age < mediana):  n={len(grupo_b)}, Fare médio={grupo_b.mean():.2f}, mediana={grupo_b.median():.2f}')

# Como Fare não é normal (Exercício 1), usaremos o teste não paramétrico Mann-Whitney U
# Também apresentamos o teste t de Student (Welch) para comparação

# Teste t de Student (Welch - variâncias desiguais)
t_stat, p_t = stats.ttest_ind(grupo_a, grupo_b, equal_var=False)
print(f'\n=== Teste t de Student (Welch) ===')
print(f'Estatística t: {t_stat:.4f}')
print(f'p-valor: {p_t:.6f}')

# Teste não paramétrico Mann-Whitney U (mais robusto para dados não-normais)
u_stat, p_mw = stats.mannwhitneyu(grupo_a, grupo_b, alternative='two-sided')
print(f'\n=== Teste Mann-Whitney U ===')
print(f'Estatística U: {u_stat:.4f}')
print(f'p-valor: {p_mw:.6f}')

print(f'\nNível de significância (α): 0.05')
if p_mw < 0.05:
    print(f'\nComo p-valor ({p_mw:.6f}) < α (0.05), rejeitamos H0.')
    print('Conclusão: Há diferença significativa no Fare entre jovens e idosos.')
else:
    print(f'\nComo p-valor ({p_mw:.6f}) >= α (0.05), não rejeitamos H0.')
    print('Conclusão: Não há diferença significativa no Fare entre jovens e idosos.')

-----------------

## EXERCÍCIO 5

Queremos verificar se homens e mulheres pagaram tarifas diferentes no Titanic.
Como a distribuição de tarifas é assimétrica e possui outliers, utilizaremos um teste não paramétrico.

Definição dos grupos

- Grupo A (Controle): Homens (Sex = 'male')

- Grupo B (Tratamento): Mulheres (Sex = 'female')

Métrica: Fare

In [0]:
# EXERCÍCIO 5: Teste Não Paramétrico - Homens vs Mulheres (Fare)
# Grupo A (Controle): Homens (Sex = 'male')
# Grupo B (Tratamento): Mulheres (Sex = 'female')
# Métrica: Fare (contínua, assimétrica com outliers)
# Teste: Mann-Whitney U (não paramétrico)

grupo_a = df[df['Sex'] == 'male']['Fare'].dropna()    # Homens
grupo_b = df[df['Sex'] == 'female']['Fare'].dropna() # Mulheres

print(f'Grupo A (Homens):   n={len(grupo_a)}, Fare médio={grupo_a.mean():.2f}, mediana={grupo_a.median():.2f}')
print(f'Grupo B (Mulheres):  n={len(grupo_b)}, Fare médio={grupo_b.mean():.2f}, mediana={grupo_b.median():.2f}')

print(f'\n--- Estatísticas Descritivas ---')
print(f'Homens   - Média: {grupo_a.mean():.2f}, Mediana: {grupo_a.median():.2f}, Desvio: {grupo_a.std():.2f}')
print(f'Mulheres - Média: {grupo_b.mean():.2f}, Mediana: {grupo_b.median():.2f}, Desvio: {grupo_b.std():.2f}')

# Teste Mann-Whitney U (não paramétrico para distribuições não-normais)
u_stat, p_value = stats.mannwhitneyu(grupo_a, grupo_b, alternative='two-sided')

print(f'\n=== Teste Mann-Whitney U ===')
print(f'Estatística U: {u_stat:.4f}')
print(f'p-valor: {p_value:.6f}')
print(f'Nível de significância (α): 0.05')

if p_value < 0.05:
    print(f'\nComo p-valor ({p_value:.6f}) < α (0.05), rejeitamos H0.')
    print('Conclusão: Homens e mulheres pagaram tarifas significativamente diferentes no Titanic.')
else:
    print(f'\nComo p-valor ({p_value:.6f}) >= α (0.05), não rejeitamos H0.')
    print('Conclusão: Não há diferença significativa nas tarifas entre homens e mulheres.')

# Visualização - Boxplot
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot([grupo_a.values, grupo_b.values], tick_labels=['Homens', 'Mulheres'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightpink')
ax.set_title('Distribuição de Fare por Sexo')
ax.set_ylabel('Fare')
plt.tight_layout()
plt.show()

------------